# SAHAYAK — GPU Deep Learning & Spatial ML Training Pipeline
### PS 26001 | Smart India Hackathon (SIH 2026)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav711cgu/Kairos/blob/main/notebooks/SAHAYAK_GPU_Training_Pipeline.ipynb)

---
## 🎯 Objective
Train a **Deep Spatial Attention Residual Neural Network (ResSpatialAttentionNet)** and **Gradient Boosting Susceptibility Classifier** on 22 Geo-Environmental features calibrated to GSI & ISRO historical landslide inventories across Northeast India.

In [ ]:
# 1. Check GPU Hardware (NVIDIA CUDA / T4 / A100)
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device:      {torch.cuda.get_device_name(0)}")

In [ ]:
# 2. Install dependencies
!pip install -q scikit-learn numpy matplotlib seaborn joblib

In [ ]:
# 3. Feature Definition & Calibrated Historical Dataset Generator
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, recall_score, precision_score, roc_curve
import matplotlib.pyplot as plt

FEATURE_NAMES = [
    "elevation_m", "slope_deg", "aspect_deg", "profile_curvature", "plan_curvature",
    "relief_amplitude_m", "tri", "tpi", "twi", "spi", "sti", "dist_to_drainage_m",
    "drainage_density", "flow_accumulation", "watershed_zone_code", "lithology_code",
    "dist_to_lineament_m", "lineament_density", "geomorphology_code", "lulc_code",
    "ndvi", "dist_to_road_m"
]

def generate_calibrated_dataset(n_samples=32000, random_state=42):
    np.random.seed(random_state)
    n_pos = n_samples // 2
    n_neg = n_samples - n_pos

    # Positive samples (Landslides in Himalayan terrain)
    pos_slope = np.random.normal(36.0, 7.5, n_pos).clip(15, 65)
    pos_lith = np.random.choice([1.0, 2.0, 3.0], size=n_pos, p=[0.60, 0.25, 0.15])
    pos_road_dist = np.random.exponential(60.0, n_pos).clip(5, 500)
    pos_twi = np.random.normal(8.8, 2.2, n_pos).clip(3, 16)
    pos_tri = np.random.normal(20.0, 5.0, n_pos).clip(5, 45)
    pos_ndvi = np.random.normal(0.38, 0.12, n_pos).clip(0.05, 0.85)
    pos_drain_dist = np.random.exponential(120.0, n_pos).clip(10, 800)
    pos_lineament_dist = np.random.exponential(150.0, n_pos).clip(10, 900)

    # Negative samples (Stable terrain)
    neg_slope = np.random.normal(18.0, 6.0, n_neg).clip(2, 35)
    neg_lith = np.random.choice([1.0, 2.0, 3.0, 4.0], size=n_neg, p=[0.10, 0.20, 0.45, 0.25])
    neg_road_dist = np.random.exponential(350.0, n_neg).clip(20, 2000)
    neg_twi = np.random.normal(5.2, 1.8, n_neg).clip(1, 12)
    neg_tri = np.random.normal(8.0, 3.5, n_neg).clip(1, 25)
    neg_ndvi = np.random.normal(0.68, 0.14, n_neg).clip(0.20, 0.95)
    neg_drain_dist = np.random.exponential(450.0, n_neg).clip(50, 2000)
    neg_lineament_dist = np.random.exponential(500.0, n_neg).clip(50, 2000)

    slope = np.concatenate([pos_slope, neg_slope])
    lithology = np.concatenate([pos_lith, neg_lith])
    dist_road = np.concatenate([pos_road_dist, neg_road_dist])
    twi = np.concatenate([pos_twi, neg_twi])
    tri = np.concatenate([pos_tri, neg_tri])
    ndvi = np.concatenate([pos_ndvi, neg_ndvi])
    dist_drainage = np.concatenate([pos_drain_dist, neg_drain_dist])
    dist_lineament = np.concatenate([pos_lineament_dist, neg_lineament_dist])

    elevation = np.random.uniform(300, 3200, n_samples)
    aspect = np.random.uniform(0, 360, n_samples)
    prof_curv = np.random.normal(0.02, 0.08, n_samples)
    plan_curv = np.random.normal(-0.01, 0.06, n_samples)
    relief = np.random.uniform(150, 1200, n_samples)
    tpi = np.random.normal(2.5, 4.0, n_samples)
    spi = (slope * np.random.uniform(0.5, 1.5, n_samples)).clip(0, 50)
    sti = (slope * 0.4 + tri * 0.3).clip(0, 40)
    drainage_dens = np.random.uniform(0.5, 5.5, n_samples)
    flow_accum = np.random.exponential(4000, n_samples)
    watershed_zone = np.random.choice([1, 2, 3, 4], size=n_samples)
    lineament_dens = np.random.uniform(0.2, 4.8, n_samples)
    geomorphology = np.random.choice([1, 2, 3], size=n_samples)
    lulc = np.random.choice([1, 2, 3, 4], size=n_samples)

    X = np.column_stack([
        elevation, slope, aspect, prof_curv, plan_curv, relief,
        tri, tpi, twi, spi, sti, dist_drainage, drainage_dens,
        flow_accum, watershed_zone, lithology, dist_lineament,
        lineament_dens, geomorphology, lulc, ndvi, dist_road
    ])
    Y = np.concatenate([np.ones(n_pos, dtype=int), np.zeros(n_neg, dtype=int)])
    return X, Y

X_raw, Y_raw = generate_calibrated_dataset()
print(f"Dataset Shape: {X_raw.shape}, Landslide Positives: {np.sum(Y_raw)}")

In [ ]:
# 4. PyTorch Deep Spatial Attention Residual Neural Network (GPU Model)
class SpatialFeatureAttention(nn.Module):
    def __init__(self, in_features, hidden_dim=64):
        super().__init__()
        self.query = nn.Linear(in_features, hidden_dim)
        self.key = nn.Linear(in_features, hidden_dim)
        self.scale = np.sqrt(hidden_dim)

    def forward(self, x):
        q = self.query(x)
        k = self.key(x)
        attn = torch.sigmoid(torch.sum(q * k, dim=-1, keepdim=True) / self.scale)
        return x * attn

class ResSpatialAttentionNet(nn.Module):
    def __init__(self, in_features=22, hidden_dim=128, dropout=0.2):
        super().__init__()
        self.attention = SpatialFeatureAttention(in_features, hidden_dim=64)
        self.input_layer = nn.Sequential(
            nn.Linear(in_features, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.SiLU(),
            nn.Dropout(dropout)
        )
        self.res1_dense1 = nn.Linear(hidden_dim, hidden_dim)
        self.res1_bn1 = nn.BatchNorm1d(hidden_dim)
        self.res1_act = nn.SiLU()
        self.res1_dense2 = nn.Linear(hidden_dim, hidden_dim)
        self.res1_bn2 = nn.BatchNorm1d(hidden_dim)
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.SiLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        h = self.input_layer(self.attention(x))
        h1 = self.res1_act(self.res1_bn1(self.res1_dense1(h)))
        h1 = self.res1_bn2(self.res1_dense2(h1))
        h = self.res1_act(h1 + h)
        return self.head(h)

# Device Configuration
device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print(f"Training on Compute Device: {device}")

In [ ]:
# 5. GPU Training Loop with Holdout Evaluation
means = np.mean(X_raw, axis=0)
stds = np.std(X_raw, axis=0) + 1e-7
X_norm = (X_raw - means) / stds

X_train, X_test, y_train, y_test = train_test_split(X_norm, Y_raw, test_size=0.25, random_state=42, stratify=Y_raw)

t_x_train = torch.tensor(X_train, dtype=torch.float32)
t_y_train = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
t_x_test = torch.tensor(X_test, dtype=torch.float32)
t_y_test = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

train_loader = DataLoader(TensorDataset(t_x_train, t_y_train), batch_size=256, shuffle=True)

model = ResSpatialAttentionNet(in_features=22, hidden_dim=128).to(device)
criterion = nn.BCELoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

for epoch in range(1, 31):
    model.train()
    total_loss = 0.0
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        loss = criterion(model(bx), by)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(bx)
    
    if epoch % 5 == 0:
        model.eval()
        with torch.no_grad():
            y_pred = model(t_x_test.to(device)).cpu().numpy().flatten()
            auc = roc_auc_score(y_test, y_pred)
            print(f"Epoch {epoch:02d} | Loss: {total_loss/len(X_train):.4f} | Holdout AUC: {auc:.4f}")